In [13]:
del llm

In [1]:
from vllm import LLM, SamplingParams

#del llm
# 替换为你的本地模型路径
#local_model_path = "/projectnb/rlhf/mingyuc/exp/qwen_1.5b/global_step_129"

#local_model_path = "/projectnb/rlhf/mingyuc/DisCO/exp/global_step_200"
#local_model_path = "/projectnb/rlhf/mingyuc/exp/qwen1.5/global_step_78"

#local_model_path = "Qwen/Qwen2.5-1.5B"
local_model_path = "/projectnb/rlhf/mingyuc/verl_github/verl/sft/global_step_587"
#local_model_path = "/projectnb/rlhf/mingyuc/verl_github/verl_main/sft/global_step_375_huggingface"

llm = LLM(model=local_model_path)

/projectnb/replearn/mingyu/anaconda/envs/verl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projectnb/replearn/mingyu/anaconda/envs/verl/lib/python3.10/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


INFO 08-19 19:30:14 [__init__.py:239] Automatically detected platform cuda.


2025-08-19 19:30:18,192	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 08-19 19:30:32 [config.py:585] This model supports multiple tasks: {'generate', 'reward', 'embed', 'score', 'classify'}. Defaulting to 'generate'.
INFO 08-19 19:30:32 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-19 19:30:34 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='/projectnb/rlhf/mingyuc/verl_github/verl/sft/global_step_587', speculative_config=None, tokenizer='/projectnb/rlhf/mingyuc/verl_github/verl/sft/global_step_587', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar', reasoning_backend=None), observability_confi

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.26s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  3.22s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  2.93s/it]



INFO 08-19 19:30:47 [loader.py:447] Loading weights took 5.95 seconds
INFO 08-19 19:30:47 [gpu_model_runner.py:1186] Model loading took 2.9105 GB and 7.011577 seconds
INFO 08-19 19:31:09 [backends.py:415] Using cache directory: /usr3/graduate/mingyuc/.cache/vllm/torch_compile_cache/b286838e33/rank_0_0 for vLLM's torch.compile
INFO 08-19 19:31:09 [backends.py:425] Dynamo bytecode transform time: 21.77 s
INFO 08-19 19:31:13 [backends.py:132] Cache the graph of shape None for later use
INFO 08-19 19:31:27 [backends.py:144] Compiling a graph for general shape takes 17.93 s
INFO 08-19 19:31:37 [monitor.py:33] torch.compile takes 39.70 s in total
INFO 08-19 19:31:38 [kv_cache_utils.py:566] GPU KV cache size: 1,154,496 tokens
INFO 08-19 19:31:38 [kv_cache_utils.py:569] Maximum concurrency for 131,072 tokens per request: 8.81x
INFO 08-19 19:31:59 [gpu_model_runner.py:1534] Graph capturing finished in 20 secs, took 0.45 GiB
INFO 08-19 19:31:59 [core.py:151] init engine (profile, create kv cache

In [2]:
import random
import collections
import heapq
from collections import deque
from typing import Tuple, List

def generate_branchy_maze(
    n: int,
    branchiness: float = 0.8,
    farthest_goal: bool = True
) -> Tuple[Tuple[int, int], Tuple[int, int], List[List[int]]]:
    # ---------- 初始化 ----------
    maze = [[1] * n for _ in range(n)]

    # start 随机挑一个偶数坐标 (保证格子间隔 2 时相邻仍在网格内)
    def rand_even(limit):                       # 0,2,4,… < limit
        max_even = limit - 1 if limit % 2 else limit - 2
        return random.randrange(0, max_even + 1, 2)

    start = (rand_even(n), rand_even(n))

    # ---------- Growing‑Tree 主循环 ----------
    def neighbors(x, y):
        # 返回: (邻居 x, 邻居 y, (wx, wy) 墙坐标增量)
        for dx, dy in [(0, 2), (2, 0), (0, -2), (-2, 0)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                yield nx, ny, dx // 2, dy // 2

    maze[start[0]][start[1]] = 0
    active = [start]

    while active:
        idx = -1 if random.random() > branchiness else random.randrange(len(active))
        x, y = active[idx]

        unvisited = [(nx, ny, wx, wy) for nx, ny, wx, wy in neighbors(x, y)
                     if maze[nx][ny] == 1]

        if unvisited:
            nx, ny, wx, wy = random.choice(unvisited)
            maze[x + wx][y + wy] = 0         # 打通墙
            maze[nx][ny] = 0
            active.append((nx, ny))
        else:
            active.pop(idx)                  # 死胡同：移除

    # ---------- 选取 goal ----------
    def bfs_farthest(src):
        """BFS 找到离 src 最远的可通行格；返回坐标"""
        vis = {src}
        q = deque([(src[0], src[1], 0)])
        far, far_dist = src, 0
        while q:
            x, y, d = q.popleft()
            if d > far_dist:
                far, far_dist = (x, y), d
            for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n and maze[nx][ny] == 0 and (nx, ny) not in vis:
                    vis.add((nx, ny))
                    q.append((nx, ny, d + 1))
        return far

    if farthest_goal:
        goal = bfs_farthest(start)
    else:
        path_cells = [(i, j) for i in range(n) for j in range(n)
                      if maze[i][j] == 0 and (i, j) != start]
        goal = random.choice(path_cells)

    # 保证 start / goal 两格都是路
    maze[start[0]][start[1]] = 0
    maze[goal[0]][goal[1]]   = 0

    return start, goal, maze


# ────────────────────────── 2. 工具函数 ──────────────────────────
DIRS = [ (0, -1), (0, 1), (-1, 0), (1, 0) ]          # 左、右、上、下
def to1(p):                                           # 0‑based → 字符串 "(x, y)" 1‑based
    return f"({p[0] + 1}, {p[1] + 1})"

def observation_str(pos, maze, goal):
    n, (x, y) = len(maze), pos
    parts = []
    for dx, dy in DIRS:                               # 左→右→上→下
        nx, ny = x + dx, y + dy
        if 0 <= nx < n and 0 <= ny < n:
            if (nx, ny) == goal:
                state = "exit"
            else:
                state = "path" if maze[nx][ny] == 0 else "wall"
        else:
            state = "wall"
        parts.append(f"({nx + 1}, {ny + 1}): {state}")
    return "Trajectory 1: " + "; ".join(parts)

# 已知网格内的 BFS 最短路（返回 deque，空表示不可达或就在原地）
def shortest_path(src, dst, walkable):
    if src == dst:
        return collections.deque()
    q = collections.deque([src])
    parent = {src: None}
    while q:
        cur = q.popleft()
        if cur == dst:
            break
        for dx, dy in DIRS:
            nxt = (cur[0] + dx, cur[1] + dy)
            if nxt in walkable and nxt not in parent:
                parent[nxt] = cur
                q.append(nxt)
    if dst not in parent:
        return collections.deque()
    path = collections.deque()
    cur = dst
    while cur != src:
        path.appendleft(cur)
        cur = parent[cur]
    return path


# ────────────────────────── 4. DEMO ──────────────────────────
if __name__ == "__main__":
    start, goal, maze = generate_branchy_maze(9)   # 或者用你自己的 maze
    print("Start:", start)
    print("End:", goal)
    for row in maze:
        print(row)


Start: (6, 8)
End: (0, 8)
[0, 0, 0, 0, 0, 0, 0, 1, 0]
[0, 1, 1, 1, 0, 1, 1, 1, 0]
[0, 0, 0, 1, 0, 0, 0, 0, 0]
[1, 1, 0, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 1, 0]
[1, 1, 1, 1, 0, 1, 1, 1, 0]
[0, 0, 0, 1, 0, 1, 0, 1, 0]
[0, 1, 0, 1, 0, 1, 0, 1, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0]


In [3]:
import re

sampling_params = SamplingParams(temperature=1.0, max_tokens=7)
tokenizer = llm.get_tokenizer()

def get_valid_moves(pos, maze, goal):
    """返回当前位置四周所有合法的1-based坐标字符串列表，如['(7, 2)', '(7, 0)', ...]"""
    DIRS = [ (0, -1), (0, 1), (-1, 0), (1, 0) ]
    n = len(maze)
    x, y = pos
    moves = []
    for dx, dy in DIRS:
        nx, ny = x + dx, y + dy
        if 1 <= nx < n+1 and 1 <= ny < n+1:
            if maze[nx - 1][ny - 1] == 0 or (nx, ny) == goal:
                moves.append(f"({nx}, {ny})")
    return moves

# 获取模型输出的坐标字符串，并提取两个数字
def extract_move(text):
    # 匹配 (多位数字, 多位数字) 格式
    match = re.search(r"\(\s*(\d{1,3})\s*,\s*(\d{1,3})\s*\)", text)
    if match:
        coord_str = match.group(0)
        row = int(match.group(1))
        col = int(match.group(2))
        return coord_str, row, col
    return None, None, None

def get_observation_chat(pos, maze, goal, index):
    """
    生成 observation 格式的 chat，返回 [{'role': 'user', 'content': ...}]
    """
    DIRS = [ (0, -1), (0, 1), (-1, 0), (1, 0) ]  # 左、右、上、下
    n = len(maze)
    x, y = pos
    parts = []
    for dx, dy in DIRS:
        nx, ny = x + dx, y + dy
        if 1 <= nx < n+1 and 1 <= ny < n+1:
            if (nx, ny) == goal:
                state = "exit"
            else:
                state = "path" if maze[nx-1][ny-1] == 0 else "wall"
        else:
            state = "wall"
        parts.append(f"({nx }, {ny }): {state}")
    obs_str = f"Trajectory {index}: " + ", ".join(parts)
    return [{'role': 'user', 'content': obs_str}]



In [4]:
from collections import deque

def max_steps_from_start(M, start, four_conn=True):
    n = len(M)
    if n == 0:
        return -1, []
    m = len(M[0])
    sx, sy = start
    if not (0 <= sx < n and 0 <= sy < m) or M[sx][sy] != 0:
        return -1, []

    dirs = [(1,0),(-1,0),(0,1),(0,-1)] if four_conn else \
           [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]

    dist = [[-1]*m for _ in range(n)]
    parent = [[None]*m for _ in range(n)]
    q = deque([(sx, sy)])
    dist[sx][sy] = 0

    while q:
        x, y = q.popleft()
        for dx, dy in dirs:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < m and M[nx][ny] == 0 and dist[nx][ny] == -1:
                dist[nx][ny] = dist[x][y] + 1
                parent[nx][ny] = (x, y)
                q.append((nx, ny))

    # 找最远点
    maxd, far = -1, None
    for i in range(n):
        for j in range(m):
            if M[i][j] == 0 and dist[i][j] > maxd:
                maxd, far = dist[i][j], (i, j)

    # 回溯路径（到达最远点的一条最短路）
    path = []
    if far is not None:
        x, y = far
        while True:
            path.append((x, y))
            if (x, y) == (sx, sy) or parent[x][y] is None:
                break
            x, y = parent[x][y]
        path.reverse()

    return maxd, path

start, goal, maze = generate_branchy_maze(9)   # 或者用你自己的 maze
maxd, path = max_steps_from_start(maze, start)
print(f"从起点 {start} 到终点 {goal} 的最远步数: {maxd}")


从起点 (4, 2) 到终点 (4, 8) 的最远步数: 18


In [65]:



#start, goal, maze = generate_branchy_maze(9)   # 或者用你自己的 maze
steps, _ = max_steps_from_start(maze, start)
print(f"从起点 {start} 到最远path的最短步数: {steps}")

# 示例用法
pos = (start[0] + 1, start[1] + 1)  # 1-based
end = (goal[0] + 1, goal[1] + 1)
counts = 0 
index = 1
chat = get_observation_chat(pos, maze, end, index)

interval_size = 25
episodes = 5

scores = 0

while True:
    if index > episodes:
        break
    chat_str = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
    chat_str = chat_str.replace(
        "<|im_start|>system\nYou are a helpful assistant.<|im_end|>",
        "<|im_start|>system\nYou are an intelligent agent navigating a maze across multiple attempts. At each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit'). Learn from previous trajectories to navigate more efficiently. Choose exactly one adjacent 'path' or 'exit' cell to move into. Output your next move as coordinates (row, col) only.<|im_end|>"
    )
    
    #chat_str = chat_str + "("
    #print("Chat string:", chat_str)
    output = llm.generate(chat_str, sampling_params)
    counts = counts + 1
    #print(output[0].outputs[0].text)

    # 假设 pos, maze, goal, chat 已定义
    valid_moves = get_valid_moves(pos, maze, end)
    model_move, x, y = extract_move( output[0].outputs[0].text)

    #print(pos, maze, goal)
    if model_move in valid_moves:
        chat.append({'role': 'assistant', 'content': model_move})
        pos = (x, y)
        if pos == end:
            #chat.append({'role': 'user', 'content': "Arrive the goal! Let's try again."})
            scores = scores + 1
            index = index + 1
            counts = 0
            pos = (start[0] + 1, start[1] + 1)
            new_observation = get_observation_chat(pos, maze, end, index)
            new_observation[0]['content'] = "Arrive the goal! Let's try again.\n\n" + new_observation[0]['content']
            chat.extend(new_observation)
        elif counts % interval_size == 0:
            #chat.append({'role': 'user', 'content': "You did not find the Exit. Let's try again."})
            index = index + 1
            counts = 0
            pos = (start[0] + 1, start[1] + 1)
            new_observation = get_observation_chat(pos, maze, end, index)
            new_observation[0]['content'] = "You did not find the Exit. Let's try again.\n\n" + new_observation[0]['content']
            chat.extend(new_observation)   
        else:         
            new_observation = get_observation_chat(pos, maze, end, index)
            chat.extend(new_observation)

    else:
        #chat.append({'role': 'user', 'content': "You did not find the Exit. Let's try again."})
        index = index + 1
        counts = 0
        pos = (start[0] + 1, start[1] + 1)
        new_observation = get_observation_chat(pos, maze, end, index)
        new_observation[0]['content'] = "You did not find the Exit. Let's try again.\n\n" + new_observation[0]['content']
        chat.extend(new_observation)  

    #print(output[0].outputs[0].text)
    #print("--------------------------")




print("Start:", (start[0] + 1, start[1] + 1))
print("End:", (goal[0] + 1, goal[1] + 1))
for row in maze:
    print(row)


print( len(chat), scores)


从起点 (2, 4) 到最远path的最短步数: 16


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 20.67it/s, est. speed input: 118125.50 toks/s, output: 144.88 toks/s]

Start: (3, 5)
End: (7, 9)
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 0, 1, 0, 1, 1, 1, 1]
[0, 0, 0, 1, 0, 0, 0, 1, 0]
[0, 1, 1, 1, 1, 1, 0, 1, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 1, 0, 1, 1, 1, 1]
[0, 1, 0, 1, 0, 0, 0, 1, 0]
[1, 1, 0, 1, 0, 1, 1, 1, 0]
[0, 0, 0, 1, 0, 0, 0, 0, 0]
215 2


In [ ]:
for ct in chat:
    print(ct['content'])

In [62]:
from copy import deepcopy



print(len(chatcopy), len(chat), scores)

133 251 0


In [ ]:

success = 0
for index in range(100):
    start, goal, maze = generate_branchy_maze(9)   # 或者用你自己的 maze
    steps, _ = max_steps_from_start(maze, start)
    print(f"从起点 {start} 到最远path的最短步数: {steps}")

    # 示例用法
    pos = (start[0] + 1, start[1] + 1)  # 1-based
    end = (goal[0] + 1, goal[1] + 1)
    counts = 0 
    index = 1
    chat = get_observation_chat(pos, maze, end, index)

    interval_size = steps + 10
    episodes = 10



    while True:
        if index > episodes:
            break
        chat_str = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
        chat_str = chat_str.replace(
            "<|im_start|>system\nYou are a helpful assistant.<|im_end|>",
            "<|im_start|>system\nYou are an intelligent agent navigating a maze across multiple attempts. At each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit'). Learn from previous trajectories to navigate more efficiently. Choose exactly one adjacent 'path' or 'exit' cell to move into. Output your next move as coordinates (row, col) only.<|im_end|>"
        )
        
        chat_str = chat_str + "("
        #print("Chat string:", chat_str)
        output = llm.generate(chat_str, sampling_params)
        counts = counts + 1
        #print(output[0].outputs[0].text)

        # 假设 pos, maze, goal, chat 已定义
        valid_moves = get_valid_moves(pos, maze, end)
        model_move, x, y = extract_move("(" + output[0].outputs[0].text)

        #print(pos, maze, goal)
        if model_move in valid_moves:
            chat.append({'role': 'assistant', 'content': model_move})
            pos = (x, y)
            if pos == end:
                chat.append({'role': 'user', 'content': "Arrive the goal! Let's try again."})
                index = index + 1
                counts = 0
                pos = (start[0] + 1, start[1] + 1)
                new_observation = get_observation_chat(pos, maze, end, index)
                chat.extend(new_observation)
            elif counts % interval_size == 0:
                chat.append({'role': 'user', 'content': "You did not find the Exit. Let's try again."})
                index = index + 1
                counts = 0
                pos = (start[0] + 1, start[1] + 1)
                new_observation = get_observation_chat(pos, maze, end, index)
                chat.extend(new_observation)   
            else:         
                new_observation = get_observation_chat(pos, maze, end, index)
                chat.extend(new_observation)

        else:
            chat.append({'role': 'user', 'content': "You did not find the Exit. Let's try again."})
            index = index + 1
            counts = 0
            pos = (start[0] + 1, start[1] + 1)
            new_observation = get_observation_chat(pos, maze, end, index)
            chat.extend(new_observation)  

        #print("(" +output[0].outputs[0].text)
        #print("--------------------------")
    
    if chat[-2]['role'] == 'user' and "Arrive the goal!" in chat[-2]['content']:
        success += 1

print(f"成功到达目标的次数: {success}")


In [46]:
import logging
from typing import Dict, Any, Optional, List, Tuple


def _process_message_tokens(
    messages: List[Dict[str, Any]],
    start_idx: int,
    end_idx: int,
    is_assistant: bool = False,
    enable_thinking: Optional[bool] = None,
    tools: Optional[List[Dict[str, Any]]] = None,
) -> Tuple[List[int], List[int], List[int]]:
    """
    Process tokens for a single message or a group of messages.

    Args:
        messages: List of message dictionaries
        start_idx: Start index in messages list
        end_idx: End index in messages list
        is_assistant: Whether this is an assistant message
        enable_thinking: Whether to enable thinking mode

    Returns:
        Tuple of (tokens, loss_mask, attention_mask)
    """
    if start_idx > 0:
        prev_applied_text = tokenizer.apply_chat_template(
            messages[:start_idx],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=enable_thinking,
            tools=tools,
        )
        if is_assistant:
            prev_applied_text_w_generation_prompt = tokenizer.apply_chat_template(
                messages[:start_idx],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
                tools=tools,
            )

    else:
        prev_applied_text = ""

    cur_applied_text = tokenizer.apply_chat_template(
        messages[:end_idx],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=enable_thinking,
        tools=tools,
    )

    #print(cur_applied_text)
    '''
    cur_applied_text = cur_applied_text.replace(
        "<|im_start|>system\\nYou are a helpful assistant.<|im_end|>",
        "<|im_start|>system\\nYou are an intelligent agent navigating a maze across multiple attempts.\\nAt each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit').\\nLearn from previous trajectories to navigate more efficiently.\\nChoose exactly one adjacent 'path' or 'exit' cell to move into.\\nOutput your next move as coordinates (row, col) only.\\n<|im_end|>"
    )
    '''
    
    # Get tokens for the current message only
    if is_assistant:
        generation_prompt_text = prev_applied_text_w_generation_prompt[len(prev_applied_text) :]
        generation_prompt_tokens = tokenizer.encode(
            generation_prompt_text,
            add_special_tokens=False,
        )
        _message_tokens = tokenizer.encode(
            cur_applied_text[len(prev_applied_text_w_generation_prompt) :],
            add_special_tokens=False,
        )
        message_tokens = generation_prompt_tokens + _message_tokens
        loss_mask = [0] * (len(generation_prompt_tokens)) + [1] * (len(message_tokens) - len(generation_prompt_tokens))
    else:
        message_tokens = tokenizer.encode(
            cur_applied_text[len(prev_applied_text) :],
            add_special_tokens=False,
        )
        loss_mask = [0] * len(message_tokens)

    attention_mask = [1] * len(message_tokens)

    return message_tokens, loss_mask, attention_mask

In [55]:
#start, goal, maze = generate_branchy_maze(5)   # 或者用你自己的 maze


enable_thinking = False


# 示例用法
pos = (start[0] + 1, start[1] + 1)  # 1-based
end = (goal[0] + 1, goal[1] + 1)
counts = 0 
index = 1
chat = get_observation_chat(pos, maze, end, index)
messages = [{'role': 'system', 'content': "You are an intelligent agent navigating a maze across multiple attempts. At each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit'). Learn from previous trajectories to navigate more efficiently. Choose exactly one adjacent 'path' or 'exit' cell to move into. Output your next move as coordinates (row, col) only."}] + chat  # 添加一个空字符串作为起始位置
messages.append({'role': 'assistant', 'content': "(5, 4)"})
concat_tokens = []
concat_loss_mask = []
concat_attention_mask = []

i = 0
while i < len(messages):
    cur_messages = messages[i]
    if cur_messages["role"] == "assistant":
        # Process assistant message
        tokens, loss_mask, attention_mask = _process_message_tokens(messages, i, i + 1, is_assistant=True, enable_thinking=enable_thinking)
        concat_tokens.extend(tokens)
        concat_loss_mask.extend(loss_mask)
        concat_attention_mask.extend(attention_mask)
        i += 1
    elif cur_messages["role"] == "tool":
        # Process consecutive tool messages
        st = i
        ed = i + 1
        while ed < len(messages) and messages[ed]["role"] == "tool":
            ed += 1
        tokens, loss_mask, attention_mask = _process_message_tokens(messages, st, ed, enable_thinking=enable_thinking)
        concat_tokens.extend(tokens)
        concat_loss_mask.extend(loss_mask)
        concat_attention_mask.extend(attention_mask)
        i = ed
    elif cur_messages["role"] in ["user", "system"]:
        # Process user or system message
        if cur_messages["role"] == "system" and i != 0:
            raise ValueError("System message should be the first message")
        tokens, loss_mask, attention_mask = _process_message_tokens(messages, i, i + 1, enable_thinking=enable_thinking)
        concat_tokens.extend(tokens)
        concat_loss_mask.extend(loss_mask)
        concat_attention_mask.extend(attention_mask)
        i += 1
    else:
        raise ValueError(f"Unknown role: {cur_messages['role']}")

In [58]:
print("Tokens:", concat_tokens)
print("Loss Mask:", concat_loss_mask)

# prompt_token_ids 已正确赋值为 concat_tokens，无需修改
prompt_token_ids = [t for t, lm in zip(concat_tokens, concat_loss_mask) if lm == 0]
print("Prompt token ids (loss_mask==0):", prompt_token_ids)

Tokens: [151644, 8948, 198, 2610, 525, 458, 24514, 8315, 59399, 264, 35096, 3941, 5248, 13553, 13, 2411, 1817, 3019, 11, 498, 5258, 458, 21930, 448, 34682, 3034, 323, 3040, 23942, 7761, 320, 34739, 488, 364, 2343, 6, 11324, 16431, 6, 11324, 13652, 1823, 14934, 504, 3681, 85548, 311, 20876, 803, 29720, 13, 22201, 6896, 825, 23942, 364, 2343, 6, 476, 364, 13652, 6, 2779, 311, 3271, 1119, 13, 9258, 697, 1790, 3271, 438, 13934, 320, 651, 11, 1375, 8, 1172, 13, 151645, 198, 151644, 872, 198, 48138, 23363, 220, 16, 25, 320, 20, 11, 220, 17, 1648, 1815, 11, 320, 20, 11, 220, 19, 1648, 1815, 11, 320, 19, 11, 220, 18, 1648, 1815, 11, 320, 21, 11, 220, 18, 1648, 7002, 151645, 198, 151644, 77091, 198, 7, 20, 11, 220, 19, 8, 151645, 198]
Loss Mask: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [62]:

print("Prompt token ids (loss_mask==0):", prompt_token_ids)

Prompt token ids (loss_mask==0): [151644, 8948, 198, 2610, 525, 458, 24514, 8315, 59399, 264, 35096, 3941, 5248, 13553, 13, 2411, 1817, 3019, 11, 498, 5258, 458, 21930, 448, 34682, 3034, 323, 3040, 23942, 7761, 320, 34739, 488, 364, 2343, 6, 11324, 16431, 6, 11324, 13652, 1823, 14934, 504, 3681, 85548, 311, 20876, 803, 29720, 13, 22201, 6896, 825, 23942, 364, 2343, 6, 476, 364, 13652, 6, 2779, 311, 3271, 1119, 13, 9258, 697, 1790, 3271, 438, 13934, 320, 651, 11, 1375, 8, 1172, 13, 151645, 198, 151644, 872, 198, 48138, 23363, 220, 16, 25, 320, 20, 11, 220, 17, 1648, 1815, 11, 320, 20, 11, 220, 19, 1648, 1815, 11, 320, 19, 11, 220, 18, 1648, 1815, 11, 320, 21, 11, 220, 18, 1648, 7002, 151645, 198, 151644, 77091, 198]


In [72]:
print("Tokens:", concat_tokens)
# 将tokens转换为文本作为input
input_text = tokenizer.decode(concat_tokens[:-7])
print("Input text preview:", input_text )

# 使用转换后的文本进行生成
output = llm.generate(input_text, sampling_params)
print("Generated output:", output[0].outputs[0].text)

Prompt token ids (loss_mask==0): [151644, 8948, 198, 2610, 525, 458, 24514, 8315, 59399, 264, 35096, 3941, 5248, 13553, 13, 2411, 1817, 3019, 11, 498, 5258, 458, 21930, 448, 34682, 3034, 323, 3040, 23942, 7761, 320, 34739, 488, 364, 2343, 6, 11324, 16431, 6, 11324, 13652, 1823, 14934, 504, 3681, 85548, 311, 20876, 803, 29720, 13, 22201, 6896, 825, 23942, 364, 2343, 6, 476, 364, 13652, 6, 2779, 311, 3271, 1119, 13, 9258, 697, 1790, 3271, 438, 13934, 320, 651, 11, 1375, 8, 1172, 13, 151645, 198, 151644, 872, 198, 48138, 23363, 220, 16, 25, 320, 20, 11, 220, 17, 1648, 1815, 11, 320, 20, 11, 220, 19, 1648, 1815, 11, 320, 19, 11, 220, 18, 1648, 1815, 11, 320, 21, 11, 220, 18, 1648, 7002, 151645, 198, 151644, 77091, 198]


Prompt token ids (loss_mask==0): [151644, 8948, 198, 2610, 525, 458, 24514, 8315, 59399, 264, 35096, 3941, 5248, 13553, 13, 2411, 1817, 3019, 11, 498, 5258, 458, 21930, 448, 34682, 3034, 323, 3040, 23942, 7761, 320, 34739, 488, 364, 2343, 6, 11324, 16431, 6, 11324, 13652, 1823, 14934, 504, 3681, 85548, 311, 20876, 803, 29720, 13, 22201, 6896, 825, 23942, 364, 2343, 6, 476, 364, 13652, 6, 2779, 311, 3271, 1119, 13, 9258, 697, 1790, 3271, 438, 13934, 320, 651, 11, 1375, 8, 1172, 13, 151645, 198, 151644, 872, 198, 48138, 23363, 220, 16, 25, 320, 20, 11, 220, 17, 1648, 1815, 11, 320, 20, 11, 220, 19, 1648, 1815, 11, 320, 19, 11, 220, 18, 1648, 1815, 11, 320, 21, 11, 220, 18, 1648, 7002, 151645, 198, 151644, 77091, 198]


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 20.25it/s, est. speed input: 2563.18 toks/s, output: 203.34 toks/s]

Prompt token ids (loss_mask==0): [151644, 8948, 198, 2610, 525, 458, 24514, 8315, 59399, 264, 35096, 3941, 5248, 13553, 13, 2411, 1817, 3019, 11, 498, 5258, 458, 21930, 448, 34682, 3034, 323, 3040, 23942, 7761, 320, 34739, 488, 364, 2343, 6, 11324, 16431, 6, 11324, 13652, 1823, 14934, 504, 3681, 85548, 311, 20876, 803, 29720, 13, 22201, 6896, 825, 23942, 364, 2343, 6, 476, 364, 13652, 6, 2779, 311, 3271, 1119, 13, 9258, 697, 1790, 3271, 438, 13934, 320, 651, 11, 1375, 8, 1172, 13, 151645, 198, 151644, 872, 198, 48138, 23363, 220, 16, 25, 320, 20, 11, 220, 17, 1648, 1815, 11, 320, 20, 11, 220, 19, 1648, 1815, 11, 320, 19, 11, 220, 18, 1648, 1815, 11, 320, 21, 11, 220, 18, 1648, 7002, 151645, 198, 151644, 77091, 198]


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 20.25it/s, est. speed input: 2563.18 toks/s, output: 203.34 toks/s]

Generated output: 







In [ ]:
# 方法3: 使用VLLM的其他API
try:
    # 检查VLLM版本和可用的API
    import vllm
    print(f"VLLM version: {vllm.__version__}")
    
    # 尝试使用不同的输入格式
    from vllm.inputs import TextPrompt
    
    # 如果TokensPrompt不工作，尝试构造一个假的prompt
    # 然后直接替换其token_ids
    class DirectTokensPrompt:
        def __init__(self, token_ids):
            self.prompt_token_ids = token_ids
            self.prompt = ""  # 空字符串
    
    direct_prompt = DirectTokensPrompt(concat_tokens)
    
    # 检查llm对象的方法
    print("Available methods in llm:", [m for m in dir(llm) if not m.startswith('_')])
    
    # 尝试不同的生成方法
    if hasattr(llm, 'generate'):
        print("Using llm.generate with DirectTokensPrompt...")
        output = llm.generate(direct_prompt, sampling_params)
        print("Generated output (method 3):", output[0].outputs[0].text)
    
except Exception as e:
    print("Method 3 failed:", e)
    
    # 最后的备用方法：使用模型的forward方法
    try:
        print("Trying direct model forward...")
        # 这需要更底层的访问
        model = llm.llm_engine.model_executor.driver_worker.model_runner.model
        with torch.no_grad():
            input_ids = torch.tensor([concat_tokens]).cuda()
            attention_mask = torch.tensor([concat_attention_mask]).cuda()
            
            # 直接调用模型
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            print("Direct model forward successful")
            
    except Exception as e2:
        print("Direct model forward failed:", e2)
        print("建议使用decode->encode方法")

In [ ]:
correctness = 0
num_trials = 0
for i in range(num_trials):



    start, goal, maze = generate_branchy_maze(14)   # 或者用你自己的 maze


    # 示例用法
    pos = (start[0] + 1, start[1] + 1)  # 1-based
    end = (goal[0] + 1, goal[1] + 1)
    chat = get_observation_chat(pos, maze, end)

    while True:
        chat_str = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
        chat_str = chat_str.replace(
            "<|im_start|>system\nYou are a helpful assistant.<|im_end|>",
            "<|im_start|>system\nYou are an intelligent agent navigating a maze.\nAt each step, you receive an observation of four adjacent cells, described by their coordinates and whether they are 'path' or 'wall'.\nYou must choose exactly one adjacent cell that is a valid 'path' or 'exit' and move into it.\nAlways move efficiently toward the goal.\nOutput your next move as a single coordinate in the format (row, col).\nDo not explain or repeat the input — just return the next move.\n<|im_end|>"
        )
        output = llm.generate(chat_str, sampling_params)



        # 假设 pos, maze, goal, chat 已定义
        valid_moves = get_valid_moves(pos, maze, end)
        model_move, x, y = extract_move(output[0].outputs[0].text)

        #print(pos, maze, goal)
        if model_move in valid_moves:
            chat.append({'role': 'assistant', 'content': model_move})
            pos = (x, y)
            new_observation = get_observation_chat(pos, maze, end)
            chat.extend(new_observation)
            if pos == end:
                correctness += 1
                break

        else:
            break





In [12]:
import copy

num_trials = 4
maze_size = 2

# 初始化每个 trial 的状态
trial_states = []
for _ in range(num_trials):
    start, goal, maze = generate_branchy_maze(maze_size)
    pos = (start[0] + 1, start[1] + 1)  # 1-based
    end = (goal[0] + 1, goal[1] + 1)
    chat = get_observation_chat(pos, maze, end)
    trial_states.append({
        "maze": maze,
        "pos": pos,
        "end": end,
        "chat": copy.deepcopy(chat),
        "done": False,
        "success": False,
        "step": 0,
    })

max_steps = 100  # 防止死循环
for step in range(max_steps):
    # 收集所有未终止 trial 的 chat
    active_indices = [i for i, t in enumerate(trial_states) if not t["done"]]
    if not active_indices:
        break

    chat_strs = []
    for i in active_indices:
        chat = trial_states[i]["chat"]
        chat_str = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
        chat_str = chat_str.replace(
            "<|im_start|>system\nYou are a helpful assistant.<|im_end|>",
            "<|im_start|>system\nYou are an intelligent agent navigating a maze.\nAt each step, you receive an observation of four adjacent cells, described by their coordinates and whether they are 'path' or 'wall'.\nYou must choose exactly one adjacent cell that is a valid 'path' or 'exit' and move into it.\nAlways move efficiently toward the goal.\nOutput your next move as a single coordinate in the format (row, col).\nDo not explain or repeat the input — just return the next move.\n<|im_end|>"
        )
        chat_strs.append(chat_str)

    # 并行推理
    outputs = llm.generate(chat_strs, sampling_params)

    # 批量处理输出
    for idx, i in enumerate(active_indices):
        t = trial_states[i]
        output_text = outputs[idx].outputs[0].text
        valid_moves = get_valid_moves(t["pos"], t["maze"], t["end"])
        model_move, x, y = extract_move(output_text)
        if model_move in valid_moves:
            t["chat"].append({'role': 'assistant', 'content': model_move})
            t["pos"] = (x, y)
            t["chat"].extend(get_observation_chat(t["pos"], t["maze"], t["end"]))
            t["step"] += 1
            if t["pos"] == t["end"]:
                t["done"] = True
                t["success"] = True
        else:
            t["done"] = True  # 非法输出，终止

# 统计正确率
num_success = sum(t["success"] for t in trial_states)
print(f"总共 {num_trials} 个迷宫，成功到达终点 {num_success} 个，正确率 {num_success / num_trials:.2%}")


Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 47.63it/s, est. speed input: 6725.10 toks/s, output: 476.89 toks/s]

总共 4 个迷宫，成功到达终点 0 个，正确率 0.00%


In [11]:
# 统计正确率
num_success = sum(t["success"] for t in trial_states)
print(f"总共 {num_trials} 个迷宫，成功到达终点 {num_success} 个，正确率 {num_success / num_trials:.2%}")

总共 100 个迷宫，成功到达终点 0 个，正确率 0.00%
